## This notebook was used in Google Colab since the dataset used was somewhat huge and any normal PC won't be able to handle the tasks.
### It is suggested to run this notebook in Cloud rather than any local machine, unless you have a powerful machine

# Imports

In [ ]:
! pip install "modin[all]" swifter numba contractions

In [ ]:
! pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
#import cudf
from numba import jit
import sys
import numpy as np
import modin.pandas as pd
import swifter
import regex as re
from sentence_transformers import SentenceTransformer
import contractions
import tqdm
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize the tools
model = SentenceTransformer('all-mpnet-base-v2', device=device)

import warnings
warnings.filterwarnings("ignore")

# Data Import - Directly importing from Kaggle

In [ ]:
import kagglehub
import bz2
import shutil
import os
import pandas as pd

# 1. Download the latest version from Kaggle
print("Downloading dataset...")
download_path = kagglehub.dataset_download("bittlingmayer/amazonreviews")

# 2. Define source and destination paths
files_to_move = ['train.ft.txt.bz2', 'test.ft.txt.bz2']
dest_folder = '/content/'

for file_name in files_to_move:
    src = os.path.join(download_path, file_name)
    dst = os.path.join(dest_folder, file_name)

    # Move files to /content folder
    shutil.copy(src, dst)
    print(f"Moved {file_name} to {dest_folder}")

    # 3. Extract the .bz2 files
    extracted_file = dst.replace('.bz2', '')
    print(f"Extracting {file_name}...")
    with bz2.open(dst, "rb") as source, open(extracted_file, "wb") as dest:
        shutil.copyfileobj(source, dest)
    print(f"Extracted to: {extracted_file}")


100%|██████████| 493M/493M [00:31<00:00, 16.3MB/s]

Extracting files...


Moved train.ft.txt.bz2 to /content/
Extracting train.ft.txt.bz2...
Extracted to: /content/train.ft.txt
Moved test.ft.txt.bz2 to /content/
Extracting test.ft.txt.bz2...
Extracted to: /content/test.ft.txt


# Data Processing

In [5]:
# 4. Example: Load a small sample into a Pandas DataFrame
# These files are large (~3.6M lines), so we'll read the first 10,000 for verification
print("\nLoading sample into Pandas...")
def load_sample(file_path, num_lines=10000):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [f.readline() for _ in range(num_lines)]

    # Split the fastText format: "__label__1 review text..."
    labels = [1 if line.split(' ')[0] == '__label__2' else 0 for line in lines]
    texts = [' '.join(line.split(' ')[1:]) for line in lines]

    return pd.DataFrame({'label': labels, 'text': texts})

train_df = load_sample('/content/train.ft.txt')
print("Train Sample Head:")
print(train_df.head())


Loading sample into Pandas...
Train Sample Head:
   label                                               text
0      1  Stuning even for the non-gamer: This sound tra...
1      1  The best soundtrack ever to anything.: I'm rea...
2      1  Amazing!: This soundtrack is my favorite music...
3      1  Excellent Soundtrack: I truly like this soundt...
4      1  Remember, Pull Your Jaw Off The Floor After He...


In [6]:
train_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   10000 non-null  int64 
 1   text    10000 non-null  object
dtypes: int64(1), object(1)
memory usage: 4.7 MB


In [7]:
def count_words(text):
    return len(str(text).split())

word_counts = train_df['text'].swifter.apply(count_words)

# Get the max value
max_val = word_counts.max()

print(f"\nMax number of words in a row: {max_val}")

Pandas Apply:   0%|          | 0/10000 [00:00<?, ?it/s]


Max number of words in a row: 212


### We need to use some model having more token capacity, since the most number of words per-review is > 200, so to be on the safe side, we use 'all-mpnet-base-v2'

In [8]:
# This will be user later on in the main program too, since the dataset is too big and cannot be processed at once
def text_process(text:str):
    if not isinstance(text, str):
        return ""

    # Expand contractions
    text = contractions.fix(text)

    # 1. Remove URLs
    text = re.sub(r'http\S+|https\S+|www\S+', '', text, flags=re.MULTILINE)

    # 2. Remove Emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)

    # 3. Keep alphanumeric characters (including unicode), spaces, and basic punctuation (.,!?)
    # Using \p{L} for Unicode letters, \p{N} for Unicode numbers.
    text = re.sub(r'[^\p{L}\p{N}\s.,!?]', '', text, flags=re.UNICODE)

    return text.strip()

In [9]:
train_df["processed_text"] = train_df["text"].swifter.apply(text_process)
train_df.head()

Pandas Apply:   0%|          | 0/10000 [00:00<?, ?it/s]

,label,text,processed_text
0,1,Stuning even for the non-gamer: This sound tra...,Stuning even for the nongamer This sound track...
1,1,The best soundtrack ever to anything.: I'm rea...,The best soundtrack ever to anything. I am rea...
2,1,Amazing!: This soundtrack is my favorite music...,Amazing! This soundtrack is my favorite music ...
3,1,Excellent Soundtrack: I truly like this soundt...,Excellent Soundtrack I truly like this soundtr...
4,1,"Remember, Pull Your Jaw Off The Floor After He...","Remember, Pull Your Jaw Off The Floor After He..."


In [11]:
embeddings = model.encode(
    train_df['processed_text'].tolist(),
    batch_size=126,
    show_progress_bar=True,
    convert_to_numpy=True # Keeps the output as a numpy array
)

# Add the embeddings to your dataframe
# We convert the numpy array to a list so it fits into a single column
train_df['embeddings'] = list(embeddings)

print(f"Embedding shape: {embeddings.shape}") # Should be (number_of_rows, 768)

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Embedding shape: (10000, 768)


### We are able to convert the texts into vectors, thereby the parts of the pipeline are ready, now it's time to assemble the pipeline

# Single Script for Processing

In [ ]:
import sys, os, re, torch, warnings
import regex as re_adv
import pandas as pd
from sentence_transformers import SentenceTransformer
import contractions
from itertools import islice
from tqdm import tqdm

# Pre-compile regex globally (Done once, not 10,000 times per chunk)
EMOJI_REGEX = re_adv.compile(r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U000024C2-\U0001F251]+", flags=re.UNICODE)
CLEAN_REGEX = re_adv.compile(r'[^\p{L}\p{N}\s.,!?]', flags=re_adv.V1)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer('all-mpnet-base-v2', device=device)
warnings.filterwarnings("ignore")

def text_process(text):
    if not isinstance(text, str): return ""
    text = contractions.fix(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = EMOJI_REGEX.sub('', text)
    return CLEAN_REGEX.sub('', text).strip()

def main(file_name, chunk_size=10000):
    out_dir = os.path.basename(file_name).replace('.ft.txt', '') + '_embeddings'
    os.makedirs(out_dir, exist_ok=True)

    # Calculate total lines to determine the "finish line" for the progress bar
    print(f"Counting lines in {file_name}...")
    with open(file_name, 'rb') as f:
        total_lines = sum(1 for _ in f)
    total_chunks = (total_lines + chunk_size - 1) // chunk_size

    with open(file_name, 'r', encoding='utf-8') as f:
        # Now tqdm knows the 'total'
        pbar = tqdm(total=total_chunks, desc="Overall Progress", unit="chunk")
        i = 0
        while True:
            lines = list(islice(f, chunk_size))
            if not lines:
                break

            data = [line.strip().split(' ', 1) for line in lines if line.strip()]
            df = pd.DataFrame(data, columns=['label', 'text'])
            df['label'] = df['label'].map(lambda x: 1 if x == '__label__2' else 0)

            processed_text = df['text'].map(text_process).tolist()

            # encode progress hidden to keep the CMD clean
            embeddings = model.encode(processed_text, batch_size=256, show_progress_bar=True)

            output_df = pd.DataFrame({'label': df['label'], 'embedding': list(embeddings)})
            output_df.to_parquet(os.path.join(out_dir, f"chunk_{i}.parquet"))

            i += 1
            pbar.update(1)
        pbar.close()

if __name__ == '__main__':
    # Using your existing logic for Colab/CMD compatibility
    input_file = '/content/test.ft.txt'
    c_size = 5000
    if len(sys.argv) > 1 and sys.argv[1] != '-f':
        input_file = sys.argv[1]
        if len(sys.argv) > 2:
            try: c_size = int(sys.argv[2])
            except: pass
    main(input_file, c_size)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Counting lines in test.ft.txt...



Overall Progress:   0%|          | 0/80 [00:00<?, ?chunk/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]


Overall Progress:   1%|▏         | 1/80 [00:45<59:56, 45.52s/chunk]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

# GPU Version

In [3]:
import sys, os, torch, warnings, tqdm
import regex as re_adv
import pandas as pd # Use standard pandas for initial CPU text work
import cudf        # Use cuDF for the final GPU-accelerated save
import contractions
from itertools import islice
from sentence_transformers import SentenceTransformer

# 1. Pre-compile Regex Globally (Huge speed boost)
EMOJI_PATTERN = re_adv.compile(r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U000024C2-\U0001F251]+", flags=re_adv.UNICODE)
CLEAN_PATTERN = re_adv.compile(r'[^\p{L}\p{N}\s.,!?]', flags=re_adv.V1)
URL_PATTERN = re_adv.compile(r'http\S+|https\S+|www\S+')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer('all-mpnet-base-v2', device=device)
warnings.filterwarnings("ignore")

def text_process(text):
    if not isinstance(text, str): return ""
    text = contractions.fix(text)
    text = URL_PATTERN.sub('', text)
    text = EMOJI_PATTERN.sub('', text)
    return CLEAN_PATTERN.sub('', text).strip()

def main(file_name, chunk_size=10000):
    out_dir = os.path.basename(file_name).replace('.ft.txt', '') + '_embeddings_data'
    os.makedirs(out_dir, exist_ok=True)

    # 2. Pre-calculate total chunks for the "Overall" Progress Bar
    print(f"Scanning {file_name} for total line count...")
    with open(file_name, 'rb') as f:
        total_lines = sum(1 for _ in f)
    total_chunks = (total_lines + chunk_size - 1) // chunk_size

    with open(file_name, 'r', encoding='utf-8') as f:
        pbar = tqdm.tqdm(total=total_chunks, desc="Overall Progress", unit="chunk")

        for i in range(total_chunks):
            lines = list(islice(f, chunk_size))
            if not lines: break

            # 3. Efficient Parsing (Split once)
            labels = [1 if l.startswith('__label__2') else 0 for l in lines]
            texts = [l.split(' ', 1)[1] if ' ' in l else "" for l in lines]

            # 4. CPU-bound cleaning (Mapped for speed)
            processed_texts = list(map(text_process, texts))

            # 5. Maximize GPU Throughput (Increased Batch Size)
            # Higher batch_size (128-256) reduces GPU idle time
            embeddings = model.encode(
                processed_texts,
                batch_size=256,
                show_progress_bar=True,
                convert_to_numpy=True
            )

            # 6. Final Data Assembly on GPU (cuDF)
            # We move data to GPU only once at the end for the Parquet write
            gdf = cudf.DataFrame({
                'label': labels,
                'embedding': list(embeddings)
            })

            gdf.to_parquet(os.path.join(out_dir, f"chunk_{i+1}.parquet"))
            pbar.update(1)

        pbar.close()
    print(f"\nSuccess! Saved to {out_dir}/")

if __name__ == '__main__':
    # Using your existing logic for Colab/CMD compatibility
    input_file = '/content/test.ft.txt'
    c_size = 5000
    if len(sys.argv) > 1 and sys.argv[1] != '-f':
        input_file = sys.argv[1]
        if len(sys.argv) > 2:
            try: c_size = int(sys.argv[2])
            except: pass
    main(input_file, c_size)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scanning /content/test.ft.txt for total line count...




Overall Progress:   0%|          | 0/80 [00:00<?, ?chunk/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Overall Progress:   2%|▎         | 1/40 [03:34<2:19:08, 214.07s/chunk]


Overall Progress:   1%|▏         | 1/80 [00:51<1:08:14, 51.82s/chunk]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Therefore we have two scripts that can be run to process the datasets, here only a few batches of data being processed is shows since you can see that each of those processes will take about an hour to finish, and you might have noticed that the GPU code's estiamated time is a bit more than that of the normal CPU or GPU code, which is because of the cudf overhead, it can be managed by some further preprocessing or modification of code.